In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/aldinwhyudii/student-depression-and-lifestyle-100k-data/student_lifestyle_100k.csv


In [40]:
import pandas as pd
import numpy as np
df=pd.read_csv("student_lifestyle_100k.csv")
df=df.drop("Student_ID",axis=1)
df.head(3)
df['Department'].unique()


array(['Science', 'Engineering', 'Medical', 'Arts', 'Business'],
      dtype=object)

In [38]:
df

,Age,CGPA,Sleep_Duration,Study_Hours,Social_Media_Hours,Physical_Activity,Stress_Level,Depression,Gender_Male,Department_Business,Department_Engineering,Department_Medical,Department_Science
0,22,3.50,7.3,3.3,3.4,114,5,False,False,False,False,False,True
1,20,2.72,5.5,7.2,6.0,142,2,False,True,False,True,False,False
2,20,3.01,5.4,2.3,1.8,137,3,False,True,False,False,True,False
3,21,3.63,8.1,2.0,4.6,130,3,False,True,False,True,False,False
4,19,3.14,6.8,2.6,4.3,4,6,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,24,2.02,7.0,0.6,1.4,66,4,False,False,False,False,True,False
99996,24,2.33,5.0,3.6,5.2,103,3,False,True,False,False,False,False
99997,24,2.23,6.5,5.8,4.1,61,5,False,False,False,True,False,False
99998,19,3.61,6.1,4.8,4.9,116,4,False,True,False,True,False,False


In [2]:
df=pd.get_dummies(df,columns=["Gender","Department"],drop_first=True)

In [3]:
y=df["Depression"]
x=df.drop("Depression",axis=1)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [5]:
x_train,x_test,y_train,y_test=train_test_split(x,y,train_size=0.9,random_state=42)

In [6]:
rf=RandomForestClassifier()
model=rf.fit(x_train,y_train)
model.score(x_test,y_test)

0.9006

In [7]:
df["Depression"].value_counts()

Depression
False    89938
True     10062
Name: count, dtype: int64

In [8]:
model.predict(x_test)

array([False, False, False, ..., False, False, False])

In [9]:
sonuc=model.predict(x_test)
dfs=pd.DataFrame()
dfs["sonuc"]=sonuc
dfs["gercek"]=y_test.values
dfs["sonuc"].value_counts()


sonuc
False    9955
True       45
Name: count, dtype: int64

In [10]:
dfs["gercek"].value_counts()

gercek
False    9009
True      991
Name: count, dtype: int64

In [11]:
dfs["truebil"]=(dfs["gercek"]==True)&(dfs["sonuc"]==True)

In [12]:
dfs["truebil"].value_counts()

truebil
False    9979
True       21
Name: count, dtype: int64

In [13]:
from sklearn.metrics import precision_score,recall_score,f1_score

In [14]:
precision=precision_score(dfs["gercek"],dfs["sonuc"])

In [18]:
precision

0.4666666666666667

In [19]:
recall=recall_score(dfs["gercek"],dfs["sonuc"])
recall

0.02119071644803229

In [16]:
2*(precision*recall)/(precision+recall)

0.040540540540540536

In [17]:
f1_score(dfs["gercek"],dfs["sonuc"])

0.04054054054054054

In [22]:
pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import xgboost as xgb

# 1. Load and clean the data
df = pd.read_csv("student_lifestyle_100k.csv")
df = df.drop("Student_ID", axis=1)
df = pd.get_dummies(df, columns=["Gender", "Department"], drop_first=True)

# --- THE UNDER-SAMPLING STEP ---
# Separate the True and False rows
df_false = df[df["Depression"] == False]
df_true = df[df["Depression"] == True]

# Randomly pick exactly the same number of 'False' rows as there are 'True' rows
df_false_downsampled = df_false.sample(n=len(df_true), random_state=42)

# Combine them back together into a new, perfectly balanced dataset
df_balanced = pd.concat([df_false_downsampled, df_true])

# Shuffle the rows so they aren't all in order
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
# -------------------------------

# 2. Separate Features (x) and Target (y) using the NEW balanced dataset
y = df_balanced["Depression"]
x = df_balanced.drop("Depression", axis=1)

# 3. Split the data (20% for testing)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 4. Train a normal XGBoost model (We don't need weights anymore because the data is 50/50!)
model = xgb.XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    max_depth=6,
    learning_rate=0.05,
    n_estimators=300
)

print("Training on the Under-sampled data...")
model.fit(x_train, y_train)

# 5. Evaluate
y_pred = model.predict(x_test)
print("\n--- Under-sampled XGBoost Evaluation ---")
print(classification_report(y_test, y_pred))

Training on the Under-sampled data...

--- Under-sampled XGBoost Evaluation ---
              precision    recall  f1-score   support

       False       0.68      0.74      0.71      2020
        True       0.71      0.64      0.67      2005

    accuracy                           0.69      4025
   macro avg       0.69      0.69      0.69      4025
weighted avg       0.69      0.69      0.69      4025



In [31]:
import joblib

# Save the model
joblib.dump(model, 'stress_model.pkl')

# Save the column names (very important because of get_dummies!)
model_columns = list(x.columns)
joblib.dump(model_columns, 'model_columns.pkl')

print("Model and columns saved successfully!")

Model and columns saved successfully!
